In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import Figure as fig
from pylab import rcParams
import datetime
from datetime import timedelta
import statsmodels.api as sm
rcParams['figure.figsize'] = 20,10
from ResearchClass import PlotEvaluations, EvaluationMetrics, TradesBook, StrategyTemplate
import os

pd.options.mode.chained_assignment = None  # default='warn'

In [ ]:
class EMAIndicator(StrategyTemplate):
    """
    EMA (Exponential Moving Average) strategy class.
    """
    def __init__(self, data, start_date, end_date, interval, indicator_parameters):
        self.data=data
        self.indicator_parameters=indicator_parameters

    def AddIndicators(self):
        """
        Adds EMA indicators to the data.
        EMA is calculated using an exponential weighting of past data.
        """
        LOOKBACK_PERIOD = self.indicator_parameters[0]  # EMA lookback period
        self.data['EMA'] = self.data['Close'].ewm(span=LOOKBACK_PERIOD, adjust=False).mean()

    def strategyLogic(self, TradeBook, row, idx):
        """
        EMA-based trading logic:
        - Buy if price crosses above EMA.
        - Sell if price crosses below EMA.
        """
        if idx < self.indicator_parameters[0]:
            return

        if TradeBook.CurrentSizing() == 0:
            if row['Close'] > row['EMA']:  # Price crosses above EMA
                TradeBook.OpenTrade(row['Open'], 1, row['Open'] * (1 - self.indicator_parameters[1]), idx)
            elif row['Close'] < row['EMA']:  # Price crosses below EMA
                TradeBook.OpenTrade(row['Open'], -1, row['Open'] * (1 + self.indicator_parameters[1]), idx)
